# AIGC 5005 · Module 01 · Part 5
## Worked example: Temperature Advisor

Companion notebook to Part 5 of the Module 01 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module01-tool` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module01-tool python=3.11 ipykernel`, then `conda activate module01-tool`. If VS Code offers to install `ipykernel`, accept.

## What this notebook does

The module's worked example is a script, `app.py`, that reads a temperature, validates it without crashing, converts it, branches on the result and reports back. The script is in `python_files/worked_example/app.py`. This notebook does two things with it:

1. Rebuilds the logic as a **function** that returns its message instead of printing and exiting, so we can try many inputs quickly.
2. Runs the **real script** in a separate process, with different typed answers, to check exits and messages.

Every exit from the program is a message, never a traceback.

In [ ]:
MIN_C = -90.0    # lowest temperature ever recorded on Earth
MAX_C = 60.0     # highest


def advise(raw):
    """Return (ok, message) for what the user typed. Mirrors app.py, without printing or exiting."""
    # Validate before converting, so a bad entry does not crash the program.
    try:
        celsius = float(raw)
    except ValueError:
        return False, f"'{raw}' is not a number. Please enter a value like 21.5"

    if not MIN_C <= celsius <= MAX_C:
        return False, (f"{celsius} is outside the plausible range "
                       f"({MIN_C} to {MAX_C}). Please check your entry.")

    fahrenheit = celsius * 9 / 5 + 32

    if celsius < 0:
        advice = "Freezing. Dress for winter."
    elif celsius < 18:
        advice = "Cold. Bring a jacket."
    elif celsius < 26:
        advice = "Comfortable."
    else:
        advice = "Hot. Stay hydrated."

    return True, f"{celsius:.1f} C is {fahrenheit:.1f} F. {advice}"

## 1. Try a range of inputs

The module suggests testing with a valid number, with text, and with something like 500. **Predict first:** which of these are accepted, and what does each message say?

In [ ]:
for raw in ["21.5", "hello", "500", "", "-10", "30", "nan", "-90", "60.1"]:
    ok, message = advise(raw)
    print(f"{raw!r:>8}  {'OK ' if ok else 'ERR'}  {message}")

## 2. Test the boundaries

The thresholds are where decision code breaks. 0, 18 and 26 sit exactly on the edges, and `-90` and `60` sit exactly on the limits of the range check.

In [ ]:
for raw in ["-0.1", "0", "17.9", "18", "25.9", "26", "-90", "60"]:
    ok, message = advise(raw)
    print(f"{raw:>6}  ->  {message}")

Read the output against the code. A temperature of exactly 0 is *Cold*, not *Freezing*, because the test is `celsius < 0`. Exactly 18 is *Comfortable*. If you change a comparison from `<` to `<=`, these are the lines that change.

## 3. Run the real script

`app.py` calls `input()`, so we run it in a separate Python process and supply the typed answer. This is what happens when someone runs `python app.py` in a terminal. We print what the script wrote, and the **exit code**: 0 for success, 1 for the two error exits.

In [ ]:
import subprocess
import sys
from pathlib import Path

script = Path("..") / "python_files" / "worked_example" / "app.py"
print("Script found:", script.exists())


def run_script(typed):
    result = subprocess.run([sys.executable, str(script)], input=typed + "\n",
                            capture_output=True, text=True)
    print(f"typed {typed!r:>8} -> exit code {result.returncode}")
    print("   ", result.stdout.strip().splitlines()[-1] if result.stdout.strip() else "(no output)")
    if result.stderr.strip():
        print("    STDERR:", result.stderr.strip().splitlines()[-1])


if script.exists():
    for typed in ["21.5", "hello", "500"]:
        run_script(typed)

No traceback in any of them: a bad entry ends with a message and exit code 1.

---
## Try it

In your own `module01-tool` folder, create `app.py` and type out the program (do not copy it). Run it with a valid number, with text, and with 500. Commit it with a message that describes what it does. Then change the comfort thresholds to values you think are more accurate, and commit that as a **separate** change. `git log --oneline` should tell a story.